In [1]:
from openai import OpenAI
import os
import getpass

os.environ["NVIDIA_API_KEY"] = getpass.getpass("Enter NVIDIA API key: ")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=os.environ["NVIDIA_API_KEY"]
)

In [2]:
MODEL = "nvidia/nemotron-3-ultra-550b-a55b"

In [4]:
prompt = """
Create a beginner-friendly practical MVP plan for a GeoAI flood monitoring workflow for Chennai, India.

Use:
- Sentinel-1 SAR data for flood detection
- Sentinel-2 optical data for flood extent mapping
- rainfall data for flood prediction
- road network data for accessibility analysis
- building footprint data for damage assessment
- drainage network data for flood risk analysis

Avoid:
- deep learning
- model fine-tuning
- A100 GPU
- complex drainage modeling
- advanced hydrological simulation
- production-scale cloud architecture

Use only:
- rule-based methods
- classical GIS analysis
- simple statistical thresholds
- beginner-friendly Python workflows

Output:
1. MVP goal
2. Simple workflow steps
3. Python libraries needed
4. Folder structure
5. JSON output schema
6. 4-week roadmap
7. What to build first in VS Code + Colab

Keep it practical for a beginner GeoAI project using Colab and VS Code.
"""

In [5]:
completion = client.chat.completions.create(
    model=MODEL,
    messages=[
        {
            "role": "system",
            "content": "You are a practical GeoAI mentor. Keep the project beginner-friendly and implementation-focused."
        },
        {
            "role": "user",
            "content": prompt
        }
    ],
    temperature=0.3,
    top_p=0.95,
    max_tokens=3000,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": True
        },
        "reasoning_budget": 2048
    },
    stream=True
)

output = ""

for chunk in completion:
    if not chunk.choices:
        continue

    text = chunk.choices[0].delta.content
    if text is not None:
        print(text, end="")
        output += text

### 1. MVP Goal
**Build a reproducible Jupyter Notebook workflow that ingests a single Sentinel-1/Sentinel-2 pair + rainfall data for a specific Chennai flood event (e.g., Nov 2021 or Dec 2023), produces a flood extent map, intersects it with roads/buildings/drainage, and exports a GeoJSON summary of impacted assets for one ward/zone.**

---

### 2. Simple Workflow Steps (Linear Script Logic)

| Step | Action | Method/Rule |
| :--- | :--- | :--- |
| **1. AOI & Time** | Define Chennai Ward boundary (GeoJSON) & Event Date Range | Manual input / `geopandas.read_file` |
| **2. SAR Flood Map** | Download S1 GRD (VV/VH) -> Calibrate -> Speckle Filter (Lee) -> Threshold | `VV < -18 dB` OR `VH < -20 dB` (Simple Otsu/Mean threshold on difference image) |
| **3. Optical Mask** | Download S2 L2A -> Cloud Mask (SCL) -> Calculate MNDWI -> Threshold | `MNDWI > 0.2` (Water) + Cloud-free pixels only |
| **4. Fusion** | Combine SAR (all-weather) + Optical (high-res) | **Logical OR**: `Flood = SAR_Water

In [11]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [12]:
from pathlib import Path

drive_path = Path("/content/drive/MyDrive/chennai_flood_mvp_plan.md")
drive_path.write_text(output, encoding="utf-8")

print(f"Saved permanently to: {drive_path}")

Saved permanently to: /content/drive/MyDrive/chennai_flood_mvp_plan.md
